## โจทย์
นักวิจัยทางชีววิทยาต้องการศึกษาความคล้ายคลึงกันทางกายภาพของนกเพนกวินในหมู่เกาะ Palmer Archipelago โดยต้องการจัดกลุ่ม (Clustering) นกเพนกวินจากลักษณะทางกายภาพ **โดยไม่ต้องพึ่งพาข้อมูลสายพันธุ์ (Species) ที่มีอยู่** เพื่อดูว่าอัลกอริทึมสามารถแบ่งกลุ่มได้ใกล้เคียงกับธรรมชาติหรือไม่

## Dataset: `penguins.csv`
ข้อมูลของนกเพนกวิน ประกอบด้วยคอลัมน์ดังนี้:
* `species`: สายพันธุ์
* `island`: ชื่อเกาะ
* `culmen_length_mm`: ความยาวจงอยปาก (มิลลิเมตร)
* `culmen_depth_mm`: ความลึกจงอยปาก (มิลลิเมตร)
* `flipper_length_mm`: ความยาวครีบ (มิลลิเมตร)
* `body_mass_g`: น้ำหนักตัว (กรัม)
* `sex`: เพศของนกเพนกวิน

## สิ่งที่ต้องทำ
1. **Data Preprocessing**
2. **Modeling & Clustering**
3. **Evaluation**
4. **สรุปผล**: สรุปว่าโมเดลใดและพารามิเตอร์ใดให้ผลลัพธ์ที่ดีที่สุด



---
### Import Library



In [50]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import pickle
from flask import Flask, request, jsonify

---
### 1. Data Preprocessing
* โหลดข้อมูล penguins.csv และสำรวจข้อมูลเบื้องต้น
* จัดการ Missing Values ด้วยวิธีที่เหมาะสม
* พิจารณาและเลือก Features ที่คิดว่ามีความจำเป็นและเหมาะสมต่อการทำ Clustering
* ทำ Data Transformation / Scaling ข้อมูลตามความจำเป็นของอัลกอริทึมที่จะเลือกใช้

In [51]:
df = pd.read_csv('penguins.csv')
df = df.dropna()

le = LabelEncoder()
df['species'] = le.fit_transform(df['species'])

X = df.drop(['species', 'island', 'sex'], axis=1)
y = df['species']
df.head(5)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)




---
### 2. Modeling & Clustering
* เลือกใช้อัลกอริทึม Clustering อย่างน้อย 3 โมเดล ที่คิดว่าเหมาะสมกับข้อมูลชุดนี้ โดยมีอย่างน้อย 1 โมเดลที่เป็น Deep Learning


In [52]:
# K-Means
input_dim = X_scaled.shape[1]

autoencoder = Sequential([
    Dense(64, activation='relu', input_shape=(input_dim,)), 
    Dense(2, activation='relu'),                          
    Dense(64, activation='relu'),                       
    Dense(input_dim, activation='linear')                   
])

autoencoder.compile(optimizer='adam', loss='mse')

history = autoencoder.fit(X_scaled, X_scaled, epochs=50, batch_size=16, verbose=0)

encoder = Model(inputs=autoencoder.layers[0].input, outputs=autoencoder.layers[1].output)
X_encoded = encoder.predict(X_scaled, verbose=0)

dl_cluster = KMeans(n_clusters=3, random_state=42)
kmeans_labels = dl_cluster.fit_predict(X_encoded)

# DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=12)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Agglomerative
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
hierarchical_labels = hc.fit_predict(X)

C:\Users\Acer\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)




---
### 3. Evaluation
* เลือกใช้ Evaluation Metrics ที่เหมาะสม 3 ตัวชี้วัด
* เขียนโค้ดเพื่อคำนวณและแสดงผลคะแนนของทุกโมเดลที่เลือกใช้


In [53]:
# Evaluate Silhouette Score
print(f"Silhouette Score: {silhouette_score(X_scaled, kmeans_labels):.4f}")
# Evaluate Calinski-Harabasz Index
print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X_scaled, kmeans_labels):.4f}")
# Evaluate Davies-Bouldin Index
print(f"Davies-Bouldin Index: {davies_bouldin_score(X_scaled, kmeans_labels):.4f}")

# Evaluate Silhouette Score
print(f"Silhouette Score: {silhouette_score(X_scaled, dbscan_labels):.4f}")
# Evaluate Calinski-Harabasz Index
print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X_scaled, dbscan_labels):.4f}")
# Evaluate Davies-Bouldin Index
print(f"Davies-Bouldin Index: {davies_bouldin_score(X_scaled, dbscan_labels):.4f}")

# Evaluate Silhouette Score
print(f"Silhouette Score: {silhouette_score(X_scaled, hierarchical_labels):.4f}")
# Evaluate Calinski-Harabasz Index
print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X_scaled, hierarchical_labels):.4f}")
# Evaluate Davies-Bouldin Index
print(f"Davies-Bouldin Index: {davies_bouldin_score(X_scaled, hierarchical_labels):.4f}")

Silhouette Score: 0.1937
Calinski-Harabasz Index: 165.3425
Davies-Bouldin Index: 1.9896
Silhouette Score: 0.3986
Calinski-Harabasz Index: 225.9199
Davies-Bouldin Index: 2.4753
Silhouette Score: 0.2702
Calinski-Harabasz Index: 188.8297
Davies-Bouldin Index: 1.4567




---
### 4. สรุปผลและบันทึก Model ที่เลือกเป็นไฟล์ `.pkl`
* เปรียบเทียบประสิทธิภาพของแต่ละโมเดลจาก Metrics ที่วัดได้
* สรุปว่าโมเดลใดและพารามิเตอร์ชุดใดเหมาะสมที่สุดสำหรับข้อมูลชุดนี้ พร้อมอธิบายเหตุผลสั้นๆ ประกอบการตัดสินใจ
* บันทึกเป็นไฟล์ .pkl


In [54]:
with open('best_model.pkl', 'wb') as f:
    pickle.dump(dbscan, f)



---
### 5. Model Deployment (REST API with Flask)

**สิ่งที่ต้องทำ:**
1. **ฝั่ง Server (Flask):** เขียนโค้ดสร้าง API endpoint `/predict` ที่รับข้อมูล JSON เพื่อทำการทำนายกลุ่ม (Cluster)
2. **ฝั่ง Client (Requests):** เขียนฟังก์ชันจำลองการส่งข้อมูล (POST Request) ไปยัง Server ของเพื่อขอผลลัพธ์

*(หมายเหตุ: ใน Colab การรัน Flask อาจจะติด Block execution ให้เขียนโค้ดเพื่อแสดง Logic การทำงานเป็นหลัก หรือใช้ `werkzeug` / `threading` หากต้องการเทสรันจริง)*


### Server

In [56]:
with open('best_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Create a new instance of the Flask class

app = Flask(__name__)

# Define a route for the default URL and its behavior
@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    features_list = [data["culmen_length_mm"], data["culmen_depth_mm"], data["flipper_length_mm"], data["body_mass_g"]]
    features_scaled = model.transform([features_list])
    prediction = model.predict(features_scaled)
    prediction = int(prediction[0])
    outdata = {'result': prediction}
    return jsonify(outdata)

    # ----------------------------------
    # YOUR CODE HERE
    # ----------------------------------

app.run(host='0.0.0.0', port=5000, debug=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.41:5000
Press CTRL+C to quit
